# Khảo sát PDF

Đặt PDF gốc trong `data/raw/`. Notebook này dành cho việc khảo sát nội dung và chất lượng trích xuất trước khi xây dựng pipeline.

Các kỹ thuật thao tác trên vấn đề liên quan đến cấu trúc văn bản sẽ dẫn tới việc tối ưu Ingestion (Chunking) và Retrieval sau này.

- Raw text extraction: Trích xuất text thủ công để kiểm tra văn bản thuộc dạng nào (searchable, ảnh (cần OCR), ..)
- Inspect blocks: Tìm hiểu cấu trúc ở tầng block. Block là một đoạn văn bản cùng cấu trúc, block có thể là header, tiêu đề chapter, tiêu đề section, một đoạn văn..
- Clean text/blocks: Thực hiện thao tác làm sạch. VD: Loại bỏ hyphenation, loại bỏ xuống dòng, loại bỏ khoảng trắng dư thừa,..
- Detect section and stateful parsing: Sử dụng regex để parser có thể hiểu được section và bước đầu parsing trạng thái. VD: Ta cần nhớ current_section = 1.1 đi qua tất cả phần body text bên dưới, cho tới khi gặp "1.2 ..." thì current_section = 1.2.
- Detect chapter: Sử dụng regex để parser nhận diện chapter. Các vấn đề: "Chapter" nằm ở header cùng page_num, "chapter" nằm trong đoạn text body -> Tận dụng regex + layout information thay vì regex đơn thuần.
- Figure/table handling: Xử lý cấu trúc dạng bảng, hình ảnh -> Lưu lại cần thiết thay vì xóa
- Footnote handling: Lưu lại các footnote lưu định nghĩa, diễn giải thay vì xóa
- Export JSON: Xem cấu trúc metadata của từng đoạn văn bản trong textbook trước khi chunking

## Path of textbook

In [ ]:
import fitz
from pathlib import Path

PDF_PATH = Path("../data/raw/Fundamentals_of_Database_System.pdf")

## Raw text extraction

In [69]:

def extract_pages(pdf_path, start_page=0, num_pages=5):
    doc = fitz.open(pdf_path)

    print(f"Total PDF pages: {len(doc)}")

    end_page = min(start_page + num_pages, len(doc))

    for page_idx in range(start_page, end_page):
        page = doc[page_idx]
        text = page.get_text("text")

        print("\n" + "=" * 80)
        print(f"PDF PAGE INDEX: {page_idx}")
        print("=" * 80)
        print(text)


extract_pages(PDF_PATH, 0, 3)

Total PDF pages: 1273

PDF PAGE INDEX: 0


PDF PAGE INDEX: 1
FUNDAMENTALS OF
Database 
Systems
SEVENTH EDITION


PDF PAGE INDEX: 2
This page intentionally left blank



In [48]:
extract_pages(PDF_PATH, start_page=34, num_pages=5)

Total PDF pages: 1273

PDF PAGE INDEX: 34
4 
Chapter 1 Databases and Database Users
store images, audio clips, and video streams digitally. These types of files are becom-
ing an important component of multimedia databases. Geographic information 
systems (GISs) can store and analyze maps, weather data, and satellite images. 
Data warehouses and online analytical processing (OLAP) systems are used in 
many companies to extract and analyze useful business information from very large 
databases to support decision making. Real-time and active database technology 
is used to control industrial and manufacturing processes. And database search 
techniques are being applied to the World Wide Web to improve the search for 
information that is needed by users browsing the Internet.
To understand the fundamentals of database technology, however, we must start 
from the basics of traditional database applications. In Section 1.1 we start by defin-
ing a database, and then we explain other basic 

## Inspect blocks

In [49]:
doc = fitz.open(PDF_PATH)

for page_idx in [34, 36]:
    page = doc[page_idx]

    print("\n" + "=" * 100)
    print(f"PAGE {page_idx}")
    print("=" * 100)

    blocks = page.get_text("blocks")

    for i, block in enumerate(blocks):
        x0, y0, x1, y1, text, block_no, block_type = block

        print(f"\nBLOCK {i}")
        print(f"bbox = ({x0:.1f}, {y0:.1f}, {x1:.1f}, {y1:.1f})")
        print(repr(text))


PAGE 34

BLOCK 0
bbox = (30.0, 29.9, 227.6, 39.3)
'4 \nChapter 1 Databases and Database Users\n'

BLOCK 1
bbox = (117.0, 62.8, 467.5, 169.4)
'store images, audio clips, and video streams digitally. These types of files are becom-\ning an important component of multimedia databases. Geographic information \nsystems (GISs) can store and analyze maps, weather data, and satellite images. \nData warehouses and online analytical processing (OLAP) systems are used in \nmany companies to extract and analyze useful business information from very large \ndatabases to support decision making. Real-time and active database technology \nis used to control industrial and manufacturing processes. And database search \ntechniques are being applied to the World Wide Web to improve the search for \ninformation that is needed by users browsing the Internet.\n'

BLOCK 2
bbox = (117.0, 176.8, 467.6, 283.4)
'To understand the fundamentals of database technology, however, we must start \nfrom the basics of 

## Check span level

In [50]:
page_dict = page.get_text("dict")

for block in page_dict["blocks"]:
    if "lines" not in block:
        continue

    for line in block["lines"]:
        for span in line["spans"]:
            print(
                span["text"],
                span["size"],
                span["font"],
                span["bbox"]
            )

6  9.0 AkzidenzGroteskBE-Md (30.0, 29.926254272460938, 37.49699783325195, 38.92625427246094)
Chapter 1 Databases and Database Users 9.0 AkzidenzGroteskBE-Regula (63.00300216674805, 30.266035079956055, 227.63101196289062, 39.26603698730469)
database management system. Of course, we are only concerned with computer- 10.5 MinionPro-Regular (117.0, 62.83780288696289, 464.9701232910156, 73.33779907226562)
ized databases in this text. 10.5 MinionPro-Regular (117.0, 74.83932495117188, 227.5125732421875, 85.33932495117188)
A  10.5 MinionPro-Regular (117.0, 92.83633422851562, 126.67050170898438, 103.33633422851562)
database management system (DBMS) 10.5 MinionPro-Bold (127.40550231933594, 92.83633422851562, 299.3042297363281, 103.33633422851562)
 is a computerized system that enables  10.5 MinionPro-Regular (299.3314514160156, 92.83633422851562, 467.539306640625, 103.33633422851562)
users to create and maintain a database. The DBMS is a  10.5 MinionPro-Regular (116.99894714355469, 104.837860107

## Clean blocks

In [51]:
import re
def clean_text(text):
    # Ghép các từ bị ngắt bởi xuống dòng:
    # "defin-\ning" -> "defining"
    text = re.sub(r"-\n(?=[a-z])", "", text)

    # Các newline còn lại -> space
    text = text.replace("\n", " ")

    # Xóa khoảng trắng thừa
    text = re.sub(r"\s+", " ", text)

    return text.strip()
text = """we start by defin-
ing a   database"""

print(clean_text(text))

we start by defining a database


In [52]:
def extract_content_blocks(page):
    blocks = page.get_text("blocks")
    page_height = page.rect.height

    results = []

    for block in blocks:
        x0, y0, x1, y1, text, block_no, block_type = block

        # bỏ image/non-text block
        if block_type != 0:
            continue

        # bỏ running header
        if y0 < 50:
            continue

        # bỏ footer rất sát cuối trang
        if y1 > page_height - 40:
            continue

        raw_text = text
        cleaned_text = clean_text(raw_text)

        if not cleaned_text:
            continue

        results.append({
            "raw_text": raw_text,
            "text": cleaned_text,
            "bbox": (x0, y0, x1, y1)
        })

    return results
blocks = extract_content_blocks(doc[34])

for block in blocks:
    print(block)

{'raw_text': 'store images, audio clips, and video streams digitally. These types of files are becom-\ning an important component of multimedia databases. Geographic information \nsystems (GISs) can store and analyze maps, weather data, and satellite images. \nData warehouses and online analytical processing (OLAP) systems are used in \nmany companies to extract and analyze useful business information from very large \ndatabases to support decision making. Real-time and active database technology \nis used to control industrial and manufacturing processes. And database search \ntechniques are being applied to the World Wide Web to improve the search for \ninformation that is needed by users browsing the Internet.\n', 'text': 'store images, audio clips, and video streams digitally. These types of files are becoming an important component of multimedia databases. Geographic information systems (GISs) can store and analyze maps, weather data, and satellite images. Data warehouses and onli

## Detect section

In [53]:
SECTION_PATTERN = re.compile(
    r"^(\d+\.\d+(?:\.\d+)*)\s+(.+)$"
)
samples = [
    "1.1 Introduction",
    "1.2 An Example",
    "5.2.1 Types of Attributes",
    "Databases and database technology"
]

for text in samples:
    match = SECTION_PATTERN.match(text)

    if match:
        print(
            "SECTION:",
            match.group(1),
            "| TITLE:",
            match.group(2)
        )
    else:
        print("BODY:", text)

SECTION: 1.1 | TITLE: Introduction
SECTION: 1.2 | TITLE: An Example
SECTION: 5.2.1 | TITLE: Types of Attributes
BODY: Databases and database technology


## Detect chapter

In [55]:
CHAPTER_PATTERN = re.compile(
    r"\bChapter\s+(\d+)\s+(.+)$",
    re.IGNORECASE
)


def detect_chapter_from_page(page):
    blocks = page.get_text("blocks")

    for block in blocks:
        x0, y0, x1, y1, text, block_no, block_type = block

        # Chỉ xét text block ở vùng header
        if block_type != 0 or y0 >= 50:
            continue

        text = clean_text(text)

        match = CHAPTER_PATTERN.search(text)

        if match:
            return {
                "number": match.group(1),
                "title": match.group(2).strip()
            }

    return None
for page_idx in range(34, 39):
    chapter = detect_chapter_from_page(doc[page_idx])

    print(page_idx, chapter)

34 {'number': '1', 'title': 'Databases and Database Users'}
35 None
36 {'number': '1', 'title': 'Databases and Database Users'}
37 None
38 {'number': '1', 'title': 'Databases and Database Users'}


In [56]:
current_chapter = None

for page_idx in range(34, 39):
    page = doc[page_idx]

    detected_chapter = detect_chapter_from_page(page)

    if detected_chapter is not None:
        current_chapter = detected_chapter

    print(
        page_idx,
        "detected =", detected_chapter,
        "| current =", current_chapter
    )

34 detected = {'number': '1', 'title': 'Databases and Database Users'} | current = {'number': '1', 'title': 'Databases and Database Users'}
35 detected = None | current = {'number': '1', 'title': 'Databases and Database Users'}
36 detected = {'number': '1', 'title': 'Databases and Database Users'} | current = {'number': '1', 'title': 'Databases and Database Users'}
37 detected = None | current = {'number': '1', 'title': 'Databases and Database Users'}
38 detected = {'number': '1', 'title': 'Databases and Database Users'} | current = {'number': '1', 'title': 'Databases and Database Users'}


In [57]:
def parse_pages(doc, start_page, end_page):
    records = []

    current_chapter = None
    current_section = None

    for page_idx in range(start_page, end_page + 1):
        page = doc[page_idx]

        # 1. Detect chapter từ header
        detected_chapter = detect_chapter_from_page(page)

        if detected_chapter is not None:
            current_chapter = detected_chapter

        # 2. Extract body blocks
        blocks = extract_content_blocks(page)

        for block in blocks:
            text = block["text"]

            # 3. Detect section
            match = SECTION_PATTERN.match(text)

            if match:
                current_section = {
                    "number": match.group(1),
                    "title": match.group(2)
                }
                continue

            # 4. Lưu body text
            records.append({
                "pdf_page": page_idx,

                "chapter_number": (
                    current_chapter["number"]
                    if current_chapter else None
                ),
                "chapter_title": (
                    current_chapter["title"]
                    if current_chapter else None
                ),

                "section_number": (
                    current_section["number"]
                    if current_section else None
                ),
                "section_title": (
                    current_section["title"]
                    if current_section else "Chapter Introduction"
                ),

                "text": text
            })

    return records
records = parse_pages(doc, 34, 38)
for r in records[:30]:
    print(
        f"PDF={r['pdf_page']} | "
        f"Chapter={r['chapter_number']} | "
        f"Section={r['section_number']} |"
        f"Section-title={r['section_title']}"
    )
    print(r["text"][:150])
    print("-" * 80)

PDF=34 | Chapter=1 | Section=None |Section-title=Chapter Introduction
store images, audio clips, and video streams digitally. These types of files are becoming an important component of multimedia databases. Geographic i
--------------------------------------------------------------------------------
PDF=34 | Chapter=1 | Section=None |Section-title=Chapter Introduction
To understand the fundamentals of database technology, however, we must start from the basics of traditional database applications. In Section 1.1 we 
--------------------------------------------------------------------------------
PDF=34 | Chapter=1 | Section=None |Section-title=Chapter Introduction
The reader who desires a quick introduction to database systems can study Sections 1.1 through 1.5, then skip or browse through Sections 1.6 through 1
--------------------------------------------------------------------------------
PDF=34 | Chapter=1 | Section=1.1 |Section-title=Introduction
Databases and database technology

## Content-type and handle figure/table

- Code thử nghiệm hiện tại chưa xử lý footnote -> Vẫn còn khá đơn giản
- Chia làm các type: normal_text, figure_caption, table_caption, table_like, footnote

In [58]:
def classify_block(text):
    text = text.strip()

    # Figure caption
    if re.match(r"^Figure\s+\d+\.\d+", text, re.IGNORECASE):
        return "figure_caption"

    # Block rất ngắn, nhiều dòng nhỏ, thường là table/diagram text
    lines = [line.strip() for line in text.split("\n") if line.strip()]

    if len(lines) >= 4:
        short_lines = sum(1 for line in lines if len(line) < 30)

        if short_lines / len(lines) > 0.7:
            return "table_like"

    return "normal_text"
blocks = doc[34].get_text("blocks")

raw_text = blocks[4][4]  # Phần tử thứ 5 của tuple là văn bản
cleaned_text = clean_text(raw_text)
block_type = classify_block(raw_text)

In [71]:
blocks = extract_content_blocks(doc[38])

for block in blocks:
    raw_text = block["raw_text"]
    cleaned_text = block["text"]

    block_type = classify_block(raw_text)

    print("TYPE:", block_type)
    print("RAW:\n", raw_text)
    print("CLEANED:", cleaned_text)
    print("-" * 80)

TYPE: table_like
RAW:
 Name
Student_number
Class
Major

CLEANED: Name Student_number Class Major
--------------------------------------------------------------------------------
TYPE: table_like
RAW:
 Smith
17
1
CS

CLEANED: Smith 17 1 CS
--------------------------------------------------------------------------------
TYPE: table_like
RAW:
 Brown
8
2
CS

CLEANED: Brown 8 2 CS
--------------------------------------------------------------------------------
TYPE: normal_text
RAW:
 STUDENT

CLEANED: STUDENT
--------------------------------------------------------------------------------
TYPE: table_like
RAW:
 Course_name
Course_number
Credit_hours
Department

CLEANED: Course_name Course_number Credit_hours Department
--------------------------------------------------------------------------------
TYPE: table_like
RAW:
 Intro to Computer Science
CS1310
4
CS

CLEANED: Intro to Computer Science CS1310 4 CS
--------------------------------------------------------------------------------
TYPE:

## Parser

In [60]:
## Version after adding content_type

def parse_pages(doc, start_page, end_page):
    records = []

    current_chapter = None
    current_section = None

    for page_idx in range(start_page, end_page + 1):
        page = doc[page_idx]

        detected_chapter = detect_chapter_from_page(page)

        if detected_chapter is not None:
            current_chapter = detected_chapter

        blocks = extract_content_blocks(page)

        for block in blocks:
            raw_text = block["raw_text"]
            cleaned_text = block["text"]

            # Detect section
            match = SECTION_PATTERN.match(cleaned_text)

            if match:
                current_section = {
                    "number": match.group(1),
                    "title": match.group(2)
                }
                continue

            # Classify content
            content_type = classify_block(raw_text)

            records.append({
                "pdf_page": page_idx,

                "chapter_number": (
                    current_chapter["number"]
                    if current_chapter else None
                ),
                "chapter_title": (
                    current_chapter["title"]
                    if current_chapter else None
                ),

                "section_number": (
                    current_section["number"]
                    if current_section else None
                ),
                "section_title": (
                    current_section["title"]
                    if current_section else "Chapter Introduction"
                ),

                "type": content_type,
                "text": cleaned_text
            })

    return records

records = parse_pages(doc, 34, 38)

from collections import Counter

print(Counter(r["type"] for r in records))

Counter({'normal_text': 55, 'table_like': 15, 'figure_caption': 2})


In [61]:
for r in records:
    if r["type"] != "normal_text":
        print(
            r["pdf_page"],
            r["section_number"],
            r["type"]
        )
        print(r["text"][:300])
        print("-" * 80)

37 1.2 figure_caption
Figure 1.1 A simplified database system environment.
--------------------------------------------------------------------------------
38 1.2 table_like
Name Student_number Class Major
--------------------------------------------------------------------------------
38 1.2 table_like
Smith 17 1 CS
--------------------------------------------------------------------------------
38 1.2 table_like
Brown 8 2 CS
--------------------------------------------------------------------------------
38 1.2 table_like
Course_name Course_number Credit_hours Department
--------------------------------------------------------------------------------
38 1.2 table_like
Intro to Computer Science CS1310 4 CS
--------------------------------------------------------------------------------
38 1.2 table_like
Data Structures CS3320 4 CS
--------------------------------------------------------------------------------
38 1.2 table_like
Discrete Mathematics MATH2410 3 MATH
--------------------

## Export JSON

In [62]:
import json
from pathlib import Path

OUTPUT_PATH = Path("../output/chapter1.jsonl")
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
    for record in records:
        f.write(
            json.dumps(
                record,
                ensure_ascii=False
            ) + "\n"
        )

In [63]:
with open(OUTPUT_PATH, "r", encoding="utf-8") as f:
    for i, line in enumerate(f):
        record = json.loads(line)

        print(record)

        if i == 4:
            break

{'pdf_page': 34, 'chapter_number': '1', 'chapter_title': 'Databases and Database Users', 'section_number': None, 'section_title': 'Chapter Introduction', 'type': 'normal_text', 'text': 'store images, audio clips, and video streams digitally. These types of files are becoming an important component of multimedia databases. Geographic information systems (GISs) can store and analyze maps, weather data, and satellite images. Data warehouses and online analytical processing (OLAP) systems are used in many companies to extract and analyze useful business information from very large databases to support decision making. Real-time and active database technology is used to control industrial and manufacturing processes. And database search techniques are being applied to the World Wide Web to improve the search for information that is needed by users browsing the Internet.'}
{'pdf_page': 34, 'chapter_number': '1', 'chapter_title': 'Databases and Database Users', 'section_number': None, 'sectio

# Test TextbookParser (from src/ingestion/textbook_parser.py)

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))
    
from src.ingestion.textbook_parser import TextbookParser


In [65]:
parser = TextbookParser(PDF_PATH)

In [66]:
records = parser.parse_pages(
    start_page=34,
    end_page=38
)

print("Total records:", len(records))

Total records: 72


In [67]:
for record in records[:10]:
    print("=" * 100)
    print("ID:", record["id"])
    print("PDF page:", record["pdf_page"])
    print(
        "Chapter:",
        record["chapter_number"],
        record["chapter_title"]
    )
    print(
        "Section:",
        record["section_number"],
        record["section_title"]
    )
    print("Type:", record["type"])
    print("Text:", record["text"][:300])

    print(
        "Footnote refs:",
        record["metadata"].get("footnote_refs", [])
    )

ID: 863264e73f7a57ea:p34:b1
PDF page: 34
Chapter: 1 Databases and Database Users
Section: None Chapter Introduction
Type: normal_text
Text: store images, audio clips, and video streams digitally. These types of files are becoming an important component of multimedia databases. Geographic information systems (GISs) can store and analyze maps, weather data, and satellite images. Data warehouses and online analytical processing (OLAP) syst
Footnote refs: []
ID: 863264e73f7a57ea:p34:b2
PDF page: 34
Chapter: 1 Databases and Database Users
Section: None Chapter Introduction
Type: normal_text
Text: To understand the fundamentals of database technology, however, we must start from the basics of traditional database applications. In Section 1.1 we start by defining a database, and then we explain other basic terms. In Section 1.2, we provide a simple UNIVERSITY database example to illustrate our
Footnote refs: []
ID: 863264e73f7a57ea:p34:b3
PDF page: 34
Chapter: 1 Databases and Database Users
S

In [ ]:
OUTPUT_PATH = PROJECT_ROOT / "output" / "chapter1_sample.jsonl"


parser = TextbookParser(PDF_PATH)

try:
    records = parser.parse_pages(34, 38)

    print("Records:", len(records))

    print(
        "Types:",
        Counter(r["type"] for r in records)
    )

    print(
        "Sections:",
        sorted({
            r["section_number"]
            for r in records
            if r["section_number"]
        })
    )

    for record in records:
        refs = record["metadata"].get(
            "footnote_refs",
            []
        )

        if refs:
            print("\nFOOTNOTE REFERENCE")
            print(record["text"][:200])
            print(refs)

    parser.export_jsonl(
        records,
        OUTPUT_PATH
    )

    print("\nSaved:", OUTPUT_PATH)

finally:
    parser.close()

Records: 72
Types: Counter({'normal_text': 46, 'table_like': 21, 'footnote': 3, 'figure_caption': 2})
Sections: ['1.1', '1.2']

FOOTNOTE REFERENCE
A database is a collection of related data. By data, we mean known facts that can be recorded and that have implicit meaning. For example, consider the names, telephone numbers, and addresses of the p
[{'marker': '1', 'raw_offset': 43, 'bbox': (292.1080017089844, 449.5342712402344, 297.6756286621094, 457.4092712402344), 'anchor_text': 'A database is a collection of related data.', 'footnote_id': '863264e73f7a57ea:p34:fn1:b8', 'status': 'resolved'}]

FOOTNOTE REFERENCE
An application program accesses the database by sending queries or requests for data to the DBMS. A query typically causes some data to be retrieved; a transaction may cause some data to be read and s
[{'marker': '2', 'raw_offset': 106, 'bbox': (229.47300720214844, 264.5343017578125, 235.0012664794922, 272.4093017578125), 'anchor_text': 'An application program accesses the data